# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains code outlining our data collection approach using the Google Maps Places API, Google Maps Routes API, and UK Police API . The full data collection pipeline is presented before the code.
</div>

In [1]:
import requests
import json
import os
from pathlib import Path
import warnings
import math
from collections import defaultdict
import re
import time

## Data Collection Pipeline

1. Gather Nightclub Data
    - Requests Google Places API using textQuery for 3 areas in London. 
    - Validates the API response to ensure that returned locations are nightclubs/bars and that their postcodes are accurate.
    - Creates a subdir called `area_name` to store the nightclub data.

2. Gather Stations Data
    - Requests Google Places API per nightclub within each area. 
    - API fieldmask limits response to 20 stations within a 1 kilometre radius from a given nightclub. 
    - Creates a subdir called `stations` within each `area_name` directory to store the stations data.

3. Filter Stations Data 
    - Divides the area surrounding the club into eight 45 degree sections. 
    - For each section, takes the two closest stations to the nightclub **as well as** always including train, subway, and transit stations.
    - Creates a subdir within `stations` called `filtered_stations` to store the filtered stations data.

4. Gather Routes Data
    - Requests Google Routes API per nightclub-station pairing to generate routes data.
    - Creates a subdir within `area_name` called `routes` to store the routes data.

5. Prepare Routes Data for Police API
    - Reduces the number of route vertices present in the Google Routes API response polyline, preserving every third coordinate only. 
    - Converts the polyline into a polygon by extending the route polyline coordinates perpendicularly by 20m each side (total corridor width of 40m).
    - Converts the polyline coordinates into an acceptable format for Police API. 
    - Creates a separate metadata json preserving metadata for each route. File names for police_params and metadata correspond to one another e.g. `amazing_grace_london_bridge_route_0_metadata.json` and `amazing_grace_london_bridge_route_0_police_api_param.txt`.
    - Creates a `police_params` and `metadata` subdir within each `area_name` to store metadata and police params.

6. Gather crime data
    - Requests UK Police API for crime data within each route corridor for six months, from 2025-01 to 2025-06. 
    - Creates a `crime_data` subdir within each `area_name` which has a subdir for each month's crime data e.g. `2025-01`

## 1. Gather nightclub data for areas in London

In [2]:
# Constants
PLACES_URL = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')
HEADERS = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': (
        'places.displayName,'
        'places.postalAddress.postalCode,'
        'places.primaryType,'
        'places.location'
    )
}

AREA_NAMES = ['Dalston', 'Hackney_Wick', 'Bermondsey']
AREA_POSTCODES = {
    'Dalston': ('E8', 'N16'),
    #https://www.townscountiespostcodes.co.uk/towns-in-england/london/dalston/
    'Hackney_Wick': ('E9', 'E15', 'E20'),
    #https://www.townscountiespostcodes.co.uk/postcodes-in-england/london/hackney-wick/
    # https://www.doogal.co.uk/AdministrativeAreas?ward=E05009374
    'Bermondsey': ('SE1', 'SE16')
    # https://www.townscountiespostcodes.co.uk/postcodes-in-england/london/bermondsey/
}


In [3]:
# define a function to normalize area names
def area_slugify(name: str) -> str:
    return re.sub(r'[\s\-]+', '_', name.strip().lower())

In [4]:
def validate_clubs(area_name):
    """
    Queries Google Places API for nightclubs within a given area, 
    validates them by checking that their primaryType is either a bar 
    or a nightclub and that they postcode is correct, caches results, 
    and returns the filtered list. It creates a subdir to store data by area.
    """

    area_slug = area_slugify(area_name)
    query_area = area_name.replace('_', '')

    cache_dir = Path(f'data/raw/{area_slug}')
    cache_path = cache_dir / (f'{area_slug}_nightclubs.json')

    try: 

        query = {
            'textQuery': f'nightclubs in {query_area}, London, England'
        }

        response = requests.post(url=PLACES_URL, headers=HEADERS, data=json.dumps(query))

        response.raise_for_status()
        clubs_data = response.json()
    
    except Exception as e:
        if cache_path.exists():
            warnings.warn(
                f'Places API failed for {area_name}, using cached data instead'
            )
            with open(cache_path) as f:
                return json.load(f)

        else:
            raise RuntimeError(
                f'API failed and no cache available for {area_name}'
            ) from e

    # Validate clubs returned from API against their primaryType and postcode

    valid_clubs = []
    postcode_prefixes = AREA_POSTCODES.get(area_name)
    if not postcode_prefixes:
        raise ValueError(f'No postcode prefixes defined for {area_name}')

    for club in clubs_data.get('places', []):
        postcode = (
            club
            .get('postalAddress', {})
            .get('postalCode', '')
        )
        primary_type = club.get("primaryType", "")

        if (
            postcode.startswith(postcode_prefixes)
            and primary_type in {"bar", "nightclub"}
        ):
            valid_clubs.append(club)

    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w') as f:
            json.dump(valid_clubs, f, indent=2)

    return valid_clubs

In [5]:
for area_name in AREA_NAMES:
    validate_clubs(area_name)

/tmp/ipykernel_610/3758504548.py:28: UserWarning: Places API failed for Dalston, using cached data instead
  warnings.warn(
/tmp/ipykernel_610/3758504548.py:28: UserWarning: Places API failed for Hackney_Wick, using cached data instead
  warnings.warn(
/tmp/ipykernel_610/3758504548.py:28: UserWarning: Places API failed for Bermondsey, using cached data instead
  warnings.warn(


## 2. Collect up to twenty stations within a radius of 1 kilometre from the nightclub

In [6]:
# Test constants

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.types,places.formattedAddress,places.location,places.displayName,places.primaryTypeDisplayName',
    }
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [7]:
# define a function to normalize club names for a coherent file system 
def slugify(name: str) -> str:
    name = name.lower()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.strip("_")

In [8]:
def get_stations_data(area_name):

    """
    Requests google places API for stations within 1000m of a nightclub.
    Stores all stations for one club in one json file
    """

    area_slug = area_slugify(area_name)

    clubs_path = Path(f'data/raw/{area_slug}/{area_slug}_nightclubs.json')

    if not clubs_path.exists():
        raise FileNotFoundError(
            f'No nightclub cache found for {area_name} at {clubs_path}'
        )
    
    with open(clubs_path, 'r') as f:
        clubs = json.load(f)

    output_dir = Path(f"data/raw/{area_slug}/stations")
    output_dir.mkdir(parents=True, exist_ok=True)

    for club in clubs:
        club_name = club['displayName']['text']
        club_slug = slugify(club_name)

        lat = round(club['location']['latitude'], ROUNDING)
        lon = round(club['location']['longitude'], ROUNDING)

        cache_path = output_dir / f'{club_slug}_stations.json'

        if cache_path.exists():
            continue

        data = {
                'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
                'maxResultCount' : 20,
                'rankPreference': 'DISTANCE',
                'locationRestriction' : {
                    'circle': {
                        'center' : {
                            'latitude' : lat,
                            'longitude' : lon
                        },
                        'radius' : 1000
                    }
                }
            }
            

        try: 
            response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data, timeout=10)
            response.raise_for_status()
            stations_data = response.json()

        except Exception as e:
            warnings.warn(
                f'Failed to fetch stations for {club_name}: {e}'
            )
            continue

        output = {
            'area': area_name,
            'club': club_name,
            'club_location': {
                'latitude': lat,
                'longitude': lon,
            },
            'places': stations_data.get('places', []),
        }

        with open(cache_path, 'w') as f:
            json.dump(output, f, indent=2)

In [9]:
for area_name in AREA_NAMES:
    get_stations_data(area_name)

## 3. Filter stations according to direction

In [10]:
# define a function that returns a bearing from point 1 to point 2 in degrees (0-360) where 0 = North and 90 = East

def bearing_degrees(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_lon = math.radians(lon2 - lon1)

    x = math.sin(delta_lon) * math.cos(phi2)
    y = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(delta_lon)
    )

    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

In [11]:
# define a function that assigns each station to one of 8 sectors

def sector_index(bearing):
    return int(bearing // 45)

In [12]:
# define haversine to calculate the shortest distance between two points on a sphere

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [13]:
def directional_filtering(stations, club_lat, club_lon, max_per_sector=2):

    """
    Divides the area around a club into 8 sectors, taking a maximum of 2 
    stations per sector. Automatically accepts train stations, subway 
    stations, and transit stations.
    """
    
    sectors = defaultdict(list)
    train_stations = []

    for station in stations['places']:
        lat = station['location']['latitude']
        lon = station['location']['longitude']

        # 1. compute straight-line distance in metres from club to station

        station['distance_m'] = haversine(club_lat, club_lon, lat, lon)

        # 2. assign station to a bearing sector relative to the club

        bearing = bearing_degrees(club_lat, club_lon, lat, lon)
        sector = sector_index(bearing)

        station['bearing'] = bearing
        station['sector'] = sector
        
        # 3. split by station type

        if station.get('primaryType') in ('train_station', 'subway_station', 'transit_station'):
            train_stations.append(station)
        else:
            sectors[sector].append(station)

    selected = []

    # select closest bus stations per sector

    for sector_stations in sectors.values():
        sector_stations.sort(key=lambda s: s['distance_m'])
        selected.extend(sector_stations[:max_per_sector])

    # always include train/subway stations

    selected_ids = {
        (
            s['location']['latitude'], s['location']['longitude']
            ) 
            for s in selected
        }

    for station in train_stations:
        key = (station['location']['latitude'], station['location']['longitude'])
        if key not in selected_ids:
            selected.append(station)

    return selected

In [14]:
def filter_stations_for_area(area_name, max_per_sector=2):
    """
    Filters stations per club per area.
    """
    
    area_slug = area_slugify(area_name)
    stations_dir = Path(f'data/raw/{area_slug}/stations')

    if not stations_dir.exists():
        raise FileNotFoundError(f'No station directory found at {stations_dir}')

    output_dir = stations_dir / 'filtered_stations'
    output_dir.mkdir(parents=True, exist_ok=True)

    for station_file in stations_dir.glob('*_stations.json'):
        with open(station_file) as f:
            data = json.load(f)

        club_lat = data['club_location']['latitude']
        club_lon = data['club_location']['longitude']

        try:
            filtered = directional_filtering(
                stations=data,
                club_lat=club_lat,
                club_lon=club_lon,
                max_per_sector=max_per_sector,
            )
        except Exception as e:
            warnings.warn(
                f'Filtering failed for {station_file.name}: {e}'
            )

            continue 

        output = {
            'area': data['area'],
            'club': data['club'],
            'club_location': data['club_location'],
            'selected_stations': filtered,
        }

        out_path = output_dir / station_file.name.replace(
            '_stations.json', '_stations_filtered.json'
        )

        with open(out_path, 'w') as f:
            json.dump(output, f, indent=2)


In [15]:
for area_name in AREA_NAMES:
    filter_stations_for_area(
        area_name = area_name,
        max_per_sector=2
    )

## 4. Generate Routes for Stations

In [16]:
# Test constants

API_KEY = os.getenv('API_KEY')
ROUTES_URL = 'https://routes.googleapis.com/directions/v2:computeRoutes'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'routes.duration,routes.distanceMeters,routes.polyline',
    }

In [17]:
def collect_routes_for_club(area_name, club_slug):

    """
    Calls google routes api for each club-station route.
    """

    area_slug = area_slugify(area_name)
    base_dir = Path(f'data/raw/{area_slug}')

    filtered_stations_dir = base_dir / 'stations' / 'filtered_stations'
    stations_path = filtered_stations_dir/ f'{club_slug}_stations_filtered.json'
    
    routes_dir = base_dir / 'routes'
    routes_dir.mkdir(parents=True, exist_ok=True)
    routes_path = routes_dir / f'{club_slug}_routes.json'

    if not stations_path.exists():
        raise FileNotFoundError(f'Missing stations file {stations_path}')

    with open(stations_path, 'r') as f:
        stations_data = json.load(f)

    club_lat = stations_data['club_location']['latitude']
    club_lon = stations_data['club_location']['longitude']

    routes_output = {
    'area': area_name,
    'club': stations_data['club'],
    'club_location': stations_data['club_location'],
    'routes': []
    }

    for station in stations_data.get('selected_stations', []):

        station_lat = station['location']['latitude']
        station_lon = station['location']['longitude']

        data = {
            'origin': {
                'location': {
                    'latLng': {
                        'latitude': club_lat,
                        'longitude': club_lon
                    }
                }
            },
            'destination': {
                'location': {
                    'latLng': {
                        'latitude': station_lat,
                        'longitude': station_lon
                    }
                }
            },
            'travelMode': 'WALK',
            'polylineEncoding': 'GEO_JSON_LINESTRING',
            'polylineQuality': 'OVERVIEW'
        }

        try:
            response = requests.post(
                ROUTES_URL,
                headers=HEADERS,
                json=data,
                timeout=10
            )
            response.raise_for_status()
            # Save one route per journey
            route = response.json()['routes'][0]

            routes_output['routes'].append({
                'station_name': station['displayName']['text'],
                'station_location': station['location'],
                'distanceMeters': route['distanceMeters'],
                'duration': route['duration'],
                'polyline': route['polyline']['geoJsonLinestring']['coordinates']
            })

        except Exception as e:
            warnings.warn(
                f'Route failed for {stations_data['club']}'
                f'{station['displayName']['text']}: {e}'
            )

    # Cache routes separately
    with open(routes_path, 'w') as f:
        json.dump(routes_output, f, indent=2)

    return routes_output

In [18]:
area_club_slugs = {}

for area_name in AREA_NAMES:
    area_slug = area_slugify(area_name)
    filtered_dir = Path(f'data/raw/{area_slug}/stations/filtered_stations')
    
    if not filtered_dir.exists():
        warnings.war(f'No filtered stations directory for {area_name}')
        continue
    
    area_club_slugs[area_name] = []

    for file_path in filtered_dir.iterdir():
        if file_path.is_file():
            club_slug = file_path.name.replace('_stations_filtered.json', '')
            area_club_slugs[area_name].append(club_slug)

for area_name, club_slugs in area_club_slugs.items():
    for club_slug in club_slugs:
        collect_routes_for_club(area_name, club_slug)

/tmp/ipykernel_610/2729615931.py:80: UserWarning: Route failed for Farr's DalstonDalston Lane /Dalston Junction Station (Stop K): 403 Client Error: Forbidden for url: https://routes.googleapis.com/directions/v2:computeRoutes
  warnings.warn(
/tmp/ipykernel_610/2729615931.py:80: UserWarning: Route failed for Farr's DalstonDalston Junction Station (Stop H): 403 Client Error: Forbidden for url: https://routes.googleapis.com/directions/v2:computeRoutes
  warnings.warn(
/tmp/ipykernel_610/2729615931.py:80: UserWarning: Route failed for Farr's DalstonDalston Junction Station (Stop G): 403 Client Error: Forbidden for url: https://routes.googleapis.com/directions/v2:computeRoutes
  warnings.warn(
/tmp/ipykernel_610/2729615931.py:80: UserWarning: Route failed for Farr's DalstonDalston Junction: 403 Client Error: Forbidden for url: https://routes.googleapis.com/directions/v2:computeRoutes
  warnings.warn(
/tmp/ipykernel_610/2729615931.py:80: UserWarning: Route failed for Farr's DalstonDalston Ju

## 5. Prepare Routes for Police API

In [19]:
def build_police_api_param(
    route,
    club_name,
    area_name,
    step=3,
    corridor_width_m=20,
):

    """
    Builds a Police API polygon param from a Google Routes API response.
    """
    # First, reduce the number of polyline coordinates for good API citizenship when querying the Police API.

    polyline = route['polyline']
    sampled = polyline[::step]

    if sampled[-1] != polyline[-1]:
        sampled.append(polyline[-1])

    # Next, build a corridor polygon from the polyline coords

    left_side = []
    right_side = []

    for i in range(len(sampled) - 1):
        # coordinates switched because Google Routes returns coords as [longitude, latitude]
        lon1, lat1 = sampled[i]
        lon2, lat2 = sampled[i + 1]

        metres_per_deg_lat = 111_320
        metres_per_deg_lon = 111_320 * math.cos(math.radians(lat1))

        dx = lon2 - lon1
        dy = lat2 - lat1

        length = math.hypot(
            dx * metres_per_deg_lon,
            dy * metres_per_deg_lat,
        )

        # extend coordinates laterally to extend the route into a corridor

        ux = -dy * metres_per_deg_lat / length
        uy =  dx * metres_per_deg_lon / length

        offset_lon = (ux * corridor_width_m) / metres_per_deg_lon
        offset_lat = (uy * corridor_width_m) / metres_per_deg_lat

        left_side.append([lat1 + offset_lat, lon1 + offset_lon])
        right_side.append([lat1 - offset_lat, lon1 - offset_lon])

    # close polygon

    lon_end, lat_end = sampled[-1]
    left_side.append([lat_end + offset_lat, lon_end + offset_lon])
    right_side.append([lat_end - offset_lat, lon_end - offset_lon])

    polygon = left_side + right_side[::-1]
    polygon.append(polygon[0])

    # Finally, edit polygon to match Police API format

    param = ":".join(f"{lat},{lon}" for lat, lon in polygon)

    return param

In [20]:
def build_route_metadata(
    *,
    area_name: str,
    club_name: str,
    club_location: dict,
    route: dict,
    route_index: int, 
    police_param_filename: str,
):
    """
    Create a metadata json for each police API param txt file.
    """
    # convert data such as 55s into int seconds 

    duration_raw = route.get("duration", "")
    duration_seconds = (
        int(duration_raw.rstrip("s"))
        if isinstance(duration_raw, str) and duration_raw.endswith("s")
        else None
    )

    metadata = {
        "area": area_name,
        "club": {
            "name": club_name,
            "location": {
                "latitude": club_location["latitude"],
                "longitude": club_location["longitude"],
            },
        },
        "destination": {
            "name": route["station_name"],
            "location": {
                "latitude": route["station_location"]["latitude"],
                "longitude": route["station_location"]["longitude"],
            },
        },
        "route": {
            "route_index": route_index,
            "distance_meters": route.get("distanceMeters"),
            "duration_seconds": duration_seconds,
        },
        "police_api": {
            "param_file": police_param_filename,
        },
    }

    return metadata

In [21]:
def batch_build_police_api_params(
    area_name,
    corridor_width_m,
):
    """
    Batch-builds Police API polygon params for all clubs in an area directory.
    Saves param as a text file, one file per route
    """

    area_dir = Path(f'data/raw/{area_name.lower()}')
    routes_dir = area_dir / 'routes'
    if not routes_dir.exists():
        raise FileNotFoundError(f'Routes directory not found: {routes_dir}')
    
    routes_files = list(routes_dir.glob("*_routes.json"))

    if not routes_files:
        raise RuntimeError(f"No route files found in {routes_dir}")

    police_params_dir = area_dir / 'police_params'
    metadata_dir = area_dir / 'metadata'

    police_params_dir.mkdir(exist_ok=True)
    metadata_dir.mkdir(exist_ok=True)

    results = {}

    for routes_path in routes_files:
        club_slug = routes_path.stem.replace("_routes", "")
        club_name = club_slug.replace("_", " ").title()

        with open(routes_path) as f:
            data = json.load(f)

        if 'routes' not in data or not data['routes']:
            warnings.warn(f'No routes found in {routes_path.name}')
            continue

        club_location = data.get('club_location')

        for i, single_route in enumerate(data['routes']):

            param_filename = f'{club_slug}_route_{i}_police_api_param.txt'
            param_path = police_params_dir/param_filename

            metadata_path = metadata_dir/f'{club_slug}_route_{i}_metadata.json'

            try:
                param = build_police_api_param(
                route=single_route,
                club_name=club_name,
                area_name=area_name,
                corridor_width_m=corridor_width_m,
                )

                param_path.write_text(param)

                metadata = build_route_metadata(
                    area_name=area_name,
                    club_name=club_name,
                    club_location=club_location,
                    route=single_route,
                    route_index=i,
                    police_param_filename=param_filename,
                )

                with open(metadata_path, 'w') as f:
                    json.dump(metadata, f, indent=2)

                results[f"{club_name} | route {i}"] = {
                    "param_file": str(param_path),
                    "metadata_file": str(metadata_path),
                }

            except Exception as e:
                warnings.warn(
                    f"Failed to build Police API param for {club_name}: {e}"
                )

    return results

In [22]:
for area_name in AREA_NAMES:
    batch_build_police_api_params(
        area_name=area_name,
        corridor_width_m=20,
    )

/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in farrs_dalston_routes.json
  warnings.warn(f'No routes found in {routes_path.name}')
/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in dalston_superstore_routes.json
  warnings.warn(f'No routes found in {routes_path.name}')
/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in ridley_road_market_bar_routes.json
  warnings.warn(f'No routes found in {routes_path.name}')
/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in copper_cats_routes.json
  warnings.warn(f'No routes found in {routes_path.name}')
/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in the_divine_routes.json
  warnings.warn(f'No routes found in {routes_path.name}')
/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in three_sheets_dalston_routes.json
  warnings.warn(f'No routes found in {routes_path.name}')
/tmp/ipykernel_610/199641430.py:36: UserWarning: No routes found in tips

## 6. Request Police API

In [23]:
def call_police_api(area_name, date):

    """
    Calls the UK Police API for each saved route polygon in an area,
    saving filtered crime data per route.
    """

    area_dir = Path(f'data/raw/{area_name.lower()}')
    params_dir = area_dir / 'police_params'

    if not params_dir.exists():
        raise FileNotFoundError(f"Area directory not found: {area_dir}")

    param_files = list(params_dir.glob("*_police_api_param.txt"))

    if not param_files:
        raise RuntimeError(f"No police API files found in {params_dir}")

    POLICE_URL = "https://data.police.uk/api/crimes-street/all-crime"

    allowed_categories = {
            "anti-social-behaviour",
            "violent-crime",
            "public-order",
            'sexual-offences',
        }

    output_dir = area_dir / 'crime_data' / date
    output_dir.mkdir(parents=True, exist_ok=True)

    for param_path in param_files:

        route_id = param_path.stem.replace('_police_api_param', '')   
        
        with open(param_path, 'r') as f:
            poly = f.read()

        params = {
            'date': date,
            'poly': poly,
        }
        
        response = requests.get(POLICE_URL, params=params)

        if response.status_code != 200:
            warnings.warn(
                f"Police API failed for {route_id}: {response.status_code}"
            )
            continue

        crimes = response.json()

        filtered_crimes = [
            c for c in crimes
            if c.get("category") in allowed_categories
        ]

        output_path = output_dir / f"{route_id}_crime_data_{date}.json"

        with open(output_path, 'w') as f:
            json.dump(filtered_crimes, f, indent=2)

        time.sleep(0.5)

In [ ]:
dates = ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06']

for area in AREA_NAMES:
    for date in dates:
        try: 
            print(f'Fetching crimes for {area} - {date}')
            call_police_api(area_name=area, date=date)
        except Exception as e:
            warnings.warn(
                f'Crime collection failed for {area} ({date}): {e}'
            )

Fetching crimes for Dalston - 2025-01
Fetching crimes for Dalston - 2025-02
Fetching crimes for Dalston - 2025-03
Fetching crimes for Dalston - 2025-04


## 7. Stations/Stops with TfL Night Service 

In [ ]:
TFL_BASE_URL = "https://api.tfl.gov.uk"

def area_slugify(name: str) -> str:
    """Normalize area names for coherent file system."""
    return name.strip().lower().replace(' ', '_').replace('-', '_')


⚠️ **Error Faced and Resolved:** 
- Initially I faced an error in that `area_slugify` was not defined. Although Eli had defined it earlier in his code, I wanted to avoid running his code as this would change the data he had already collected and would require me getting my own Google API key. I learnt this from a previous mistake I made in which I ran all of NB01 again once he had handed it over to me and the re-running had failed. Thus to solve this error I made Section 7 completely self-contained. Given the `area_slugify` function is so simple (just converts names to lowercase with underscores) it was fine to define it twice, and the function does the same thing in the Section 1 and Section 7 so there is no conflict. 

💭 **Personal Reflection:**
- a `slug` is a URL friendly string or text and to `slugify` is to convert a text or string like a title or phrase into a `slug` by making it lowercase, removing special characters and replacing spaces with hyphens.

- While Eli has used caching in his API calls above, I have not used caching here because TfL API does not require an API key (unlike Google Places API), has simple endpoints and data changes more, with night service routes being added/removed, compared to nightclub locations which are stable.

In [ ]:
def check_station_has_night_service(station_lat, station_lon):
   # Find TfL stop points near this station
    url = f"{TFL_BASE_URL}/StopPoint"
    params = {
        'lat': station_lat,
        'lon': station_lon,
        'stopTypes': 'NaptanMetroStation,NaptanRailStation,NaptanBusCoachStation,NaptanPublicBusCoachTram',
        'radius': 150
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        if '429' in str(e):
            warnings.warn("Rate limited, waiting 5 seconds...")
            time.sleep(5)
            try:
                response = requests.get(url, params=params, timeout=10)
                response.raise_for_status()
                data = response.json()
            except Exception as retry_e:
                warnings.warn(f"Failed after retry: {retry_e}")
                return False
        else:
            warnings.warn(f"Failed to find stop points: {e}")
            return False
    except Exception as e:
        warnings.warn(f"Failed to find stop points: {e}")
        return False
    
    if not data or 'stopPoints' not in data or len(data['stopPoints']) == 0:
        return False
    
    # Known night tube lines (https://www.london.gov.uk/transport/rail-and-underground/night-tube)
    night_tube_lines = ['central', 'victoria', 'jubilee', 'northern', 'piccadilly']
    
    # Complete list of 24-hour bus routes (from TfL - December  - https://bus-routes-in-london.fandom.com/wiki/List_of_24-Hour_Routes)
    known_24hr = [
        '6', '12', '13', '14', '23', '24', '35', '36', '37', '43', '47', '52', '57', 
        '64', '69', '76', '85', '88', '93', '94', '102', '105', '108', '111', '119', '123', 
        '128', '134', '139', '148', '149', '158', '159', '176', '188', '189', '213', '214', 
        '220', '222', '238', '242', '243', '264', '277', '281', '285', '295', '297', '321', 
        '341', '344', '345', '365', '390', '453', '472', '474', '486', 'EL1'
    ]
    
    # Check lines at each stop point
    for stop in data['stopPoints']:
        lines = stop.get('lines', [])
        
        for line in lines:
            line_name = line.get('name', '')
            mode = line.get('modeName', '').lower()
            
            if not line_name:
                continue
            
            # Night tube check
            if mode == 'tube' and line_name.lower() in night_tube_lines:
                return True
            
            # Night bus check (N-prefix)
            if line_name.startswith('N'):
                return True
            
            # 24-hour bus check
            if line_name in known_24hr:
                return True
    
    return False


💭 **Personal Reflection:**

- Google API produces station coordinates (lat/lon) but TfL needs a stop point ID to query night service. I introduced a radius of 150 metres as a station may be recorded as 51.5050 in our data and 51.5052 in TfL. This way we avoid missing data because of variations in the APIs. Additionally, a single station, such as London Bridge, may have multiple stop points (tube platforms, bus stops, etc.). The radius will capture these additional stop points.

⚠️ **Error Faced and Resolved:** 
- With my intial approach, only 13 out of 194 routes were found to be viable and I faced the error "429 Client Error" (Rate limiting). I increased the delays to 0.5 seconds between checking different stops at the same station and 2 seconds between checking different routes. If it hits a 429 error it will automatically retrie with a 5 second wait. The final implementation successfully identified 159 viable routes (82.0%) and 35 non-viable routes (18%); the non-viable routes will be filtered out in NB02 (Data Transformation). 

In [ ]:
def analyse_route_night_viability(area_slug, club_slug, route_index):
    """
    Check if a specific route has night service, and return a dictionary with route viability (True/False)
    """
    routes_path = Path(f'data/raw/{area_slug}/routes/{club_slug}_routes.json')
    
    with open(routes_path, 'r') as f:
        routes_data = json.load(f)
    
    route = routes_data['routes'][route_index]
    station_name = route.get('station_name', '')
    station_location = route.get('station_location', {})
    
    station_lat = station_location['latitude']
    station_lon = station_location['longitude']
    
    # Check if this station has night service
    has_night_service = check_station_has_night_service(station_lat, station_lon)
    
    return {
        'area': area_slug,
        'club': routes_data.get('club', ''),
        'club_location': routes_data.get('club_location', {}),
        'route_index': route_index,
        'station_name': station_name,
        'station_location': station_location,
        'route_viable': has_night_service
    }


In [ ]:
def collect_night_service_for_area(area_slug):
    """
    Collect night service data for all routes in an area.
    """
    routes_dir = Path(f'data/raw/{area_slug}/routes')
    
    if not routes_dir.exists():
        return {'total': 0, 'viable': 0, 'not_viable': 0}
    
    night_service_dir = Path(f'data/raw/{area_slug}/night_service')
    night_service_dir.mkdir(parents=True, exist_ok=True)
    
    route_files = list(routes_dir.glob('*_routes.json'))
    
    total_routes = 0
    viable_routes = 0
    not_viable_routes = 0
    
    for route_file in route_files:
        club_slug = route_file.stem.replace('_routes', '')
        
        with open(route_file, 'r') as f:
            routes_data = json.load(f)
        
        num_routes = len(routes_data.get('routes', []))
        total_routes += num_routes
        
        print(f"    Club: {club_slug}")
        
        for i in range(num_routes):
            print(f"      Route {i}: {routes_data['routes'][i].get('station_name', 'Unknown')}...")
            
            analysis = analyse_route_night_viability(area_slug, club_slug, i)
            
            if analysis:
                output_path = night_service_dir / f'{club_slug}_route_{i}_night_service.json'
                with open(output_path, 'w') as f:
                    json.dump(analysis, f, indent=2)
                
                if analysis['route_viable']:
                    viable_routes += 1
                    print(f"        VIABLE")
                else:
                    not_viable_routes += 1
                    print(f"        NOT VIABLE")
            
            time.sleep(2)
    
    return {
        'total': total_routes,
        'viable': viable_routes,
        'not_viable': not_viable_routes
    }


In [ ]:
# Execute for all areas
AREA_NAMES = ['Dalston', 'Hackney_Wick', 'Bermondsey']

print("7. Check Night Service for all routes...")
print()

overall_stats = {'total': 0, 'viable': 0, 'not_viable': 0}

for area_name in AREA_NAMES:
    area_slug = area_slugify(area_name)
    
    print(f"📍 Area: {area_name}")
    
    stats = collect_night_service_for_area(area_slug)
    
    overall_stats['total'] += stats['total']
    overall_stats['viable'] += stats['viable']
    overall_stats['not_viable'] += stats['not_viable']
    
    print(f"    Total: {stats['total']}, Viable: {stats['viable']}, Not viable: {stats['not_viable']}")

print(f"Overall: {overall_stats['total']} routes total")
print(f"   Viable: {overall_stats['viable']} ({overall_stats['viable']/overall_stats['total']*100:.1f}%)")
print(f"   Non-viable: {overall_stats['not_viable']} ({overall_stats['not_viable']/overall_stats['total']*100:.1f}%)")

7. Check Night Service for all routes...

📍 Area: Dalston
    Club: farrs_dalston
      Route 0: Dalston Lane /Dalston Junction Station (Stop K)...
        VIABLE
      Route 1: Dalston Junction Station (Stop H)...
        VIABLE
      Route 2: Dalston Junction Station (Stop G)...
        VIABLE
      Route 3: Dalston Junction...
        VIABLE
      Route 4: Dalston Junction Station (Stop M)...
        VIABLE
      Route 5: Kingsland High Street / Dalston Junction (Stop L)...
        VIABLE
      Route 6: Dalston Kingsland...
        VIABLE
      Route 7: Dalston Kingsland Station (Stop P)...
        VIABLE
      Route 8: Shacklewell Lane Dalston (Stop X)...
        VIABLE
      Route 9: Haggerston...
        VIABLE
      Route 10: Haggerston Station Downham Rd...
        VIABLE
      Route 11: Hackney Downs Station (Stop N)...
        NOT VIABLE
      Route 12: Hackney Downs...
        NOT VIABLE
    Club: dalston_superstore
      Route 0: Rio Cinema (Stop T)...
        VIABLE
      

💭 **Personal Reflection:**
- I added Section 7 to NB01 to colelct data on night service availability for each route, as routes to stations/bus stops without Night Tube or Night Bus service would be pointless for nightclub-goers returning home in the early hours. I queried the TfL API to check whether stations have night service. 

- The initial approach used TfL's `serviceTypes=Night` parameter to filter routes, but this proved unreliable as it returned zero results even for stops that definitely do have ngiht service (e.g. Dalston Junction which serves teh N242, N277 and N38). After debugging, I discovered that TfL's API response often had missing `modeName` fields, and more importantly that the `serviceTypes` filter was not capturing 24-hour bus routes. I, therefore, decided to match `/StopPoint` line names against known night services: Night Tuble lines (Central, Victoria, Jubilee, Norhtern, Piccadilly), any route starting with 'N' (night buses) and a list of 24-hour bus routes from. I got this information: night tubes - https://www.london.gov.uk/transport/rail-and-underground/night-tube; 24 hour buses - https://bus-routes-in-london.fandom.com/wiki/List_of_24-Hour_Routes. 

- I initially included a second stage  that made additional `/Route API` calls for thoroughness, but after verifying that the initial stopPoints data was accurate, I removed this to improve efficiency and avoid unnecessary API calls. 